# RSNA Spleen Viewer - CT + Mask + GradCAM + Metadata
GradCAM is embedded into full DICOM volume at correct crop bbox position.

In [1]:
!pip install dicomsdl

   ---------------------------------------- 0.0/923.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/923.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/923.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/923.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/923.9 kB ? eta -:--:--
   --------------------- ---------------- 524.3/923.9 kB 356.7 kB/s eta 0:00:02
   --------------------- ---------------- 524.3/923.9 kB 356.7 kB/s eta 0:00:02
   -------------------------------- ----- 786.4/923.9 kB 459.5 kB/s eta 0:00:01
   -------------------------------------- 923.9/923.9 kB 492.4 kB/s eta 0:00:00


In [2]:
import os, json
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import pydicom
import nibabel as nib
import dicomsdl
import pandas as pd
import uuid
import cv2
from scipy.ndimage import zoom
from ipywidgets import IntSlider, HBox, Button, Output, Dropdown, VBox
from IPython.display import FileLink, display

ModuleNotFoundError: No module named 'cv2'

In [ ]:
ROOT_PATH    = "/kaggle/input/competitions/rsna-2023-abdominal-trauma-detection"
MASKS_PATH   = "/kaggle/input/datasets/bmohamedelamine/rsna-2023-spleen-masks"
GRADCAM_DIR  = "/kaggle/input/datasets/bmohamedelamine/rsna-spleen-cls-gcam-masks/gradcam_volumes"
SERIES_CSV   = "/kaggle/input/datasets/bmohamedelamine/rsna-spleen-cls-labels/spleen_cls_train_series.csv"
CROP_META    = "/kaggle/input/datasets/bmohamedelamine/rsna-2023-cropped-spleen/cropped_spleen/crop_metadata.json"

In [ ]:
ROOT_PATH    = "J:\rsna-2023-abdominal-trauma-detection"
MASKS_PATH   = "/kaggle/input/datasets/bmohamedelamine/rsna-2023-spleen-masks"
GRADCAM_DIR  = "/kaggle/input/datasets/bmohamedelamine/rsna-spleen-cls-gcam-masks/gradcam_volumes"
SERIES_CSV   = "/kaggle/input/datasets/bmohamedelamine/rsna-spleen-cls-labels/spleen_cls_train_series.csv"
CROP_META    = "/kaggle/input/datasets/bmohamedelamine/rsna-2023-cropped-spleen/cropped_spleen/crop_metadata.json"

In [ ]:
# ================================================================
#  DICOM helpers
# ================================================================

def __dataset__to_numpy_image(self, index=0):
    info = self.getPixelDataInfo()
    dtype = info["dtype"]
    if info["SamplesPerPixel"] != 1:
        raise RuntimeError("SamplesPerPixel != 1")
    shape = [info["Rows"], info["Cols"]]
    arr = np.empty(shape, dtype=dtype)
    self.copyFrameData(index, arr)
    return arr

if not hasattr(dicomsdl._dicomsdl.DataSet, "to_numpy_image"):
    dicomsdl._dicomsdl.DataSet.to_numpy_image = __dataset__to_numpy_image


def fast_glob_sorted(path):
    return sorted(glob(path), key=lambda x: int(os.path.basename(x).split(".")[0]))


def fast_window(img, center, width):
    low = center - width / 2
    high = center + width / 2
    img = np.clip(img, low, high)
    return (img - low) / (high - low + 1e-6)


def load_dicom_series_fast(series_dir, center, width):
    paths = fast_glob_sorted(os.path.join(series_dir, "*.dcm"))
    if not paths:
        raise FileNotFoundError("No DICOM files found")
    volume = []
    for p in paths:
        dcm = dicomsdl.open(p)
        img = dcm.to_numpy_image().astype(np.float32)
        slope = getattr(dcm, "RescaleSlope", 1.0)
        intercept = getattr(dcm, "RescaleIntercept", 0.0)
        img = img * slope + intercept
        img = fast_window(img, center, width)
        volume.append(img)
    return np.stack(volume).astype(np.float32)

In [ ]:
# ================================================================
#  Mask loader
# ================================================================

def load_mask(mask_path, volume_shape):
    import shutil
    tmp = os.path.join("/kaggle/working", "tmp_mask.nii.gz")
    shutil.copy(mask_path, tmp)
    mask_nii = nib.load(tmp)
    mask_data = mask_nii.get_fdata()
    mask_data = np.transpose(mask_data, (2, 1, 0))
    mask_data = mask_data[::-1, :, :]
    factors = np.array(volume_shape) / np.array(mask_data.shape)
    resized = zoom(mask_data, factors, order=0)
    return (resized > 0.5).astype(np.uint8)

In [ ]:
# ================================================================
#  GradCAM helpers  (Option A: embed into full volume via crop bbox)
# ================================================================

CLASS_NAMES = ["Healthy", "Low Damage", "High Damage"]


def _load_crop_metadata(crop_meta_path):
    """Load crop_metadata.json and index by (pid, sid)."""
    if not os.path.isfile(crop_meta_path):
        print(f"  crop_metadata.json not found at {crop_meta_path}")
        return {}
    with open(crop_meta_path) as f:
        entries = json.load(f)
    lookup = {}
    for entry in entries:
        key = (str(entry["patient_id"]), str(entry["series_id"]))
        lookup[key] = entry
    return lookup


def load_gradcam(gradcam_dir, crop_lookup, pid, sid, full_volume_shape):
    """
    Load GradCAM .npz (whose cam matches the CROPPED spleen region)
    and embed it into a zero-filled array of full_volume_shape using
    the crop_bbox from crop_metadata.json.
    """
    npz_path = os.path.join(gradcam_dir, f"{pid}_{sid}.npz")
    if not os.path.isfile(npz_path):
        return None, None, None, None

    # Look up crop bbox
    meta_entry = crop_lookup.get((str(pid), str(sid)))
    if meta_entry is None:
        print(f"  No crop metadata for {pid}_{sid}, skipping GradCAM")
        return None, None, None, None

    try:
        g = np.load(npz_path)
        cam_crop = g["cam"].astype(np.float32)
        probs = g["probs"]
        pred = int(g["pred"])
        true_label = int(g["true_label"])

        # Normalize the cropped cam to [0, 1]
        lo, hi = cam_crop.min(), cam_crop.max()
        if hi > lo:
            cam_crop = (cam_crop - lo) / (hi - lo)
        else:
            cam_crop = np.zeros_like(cam_crop)

        # Get crop bbox coordinates
        bbox = meta_entry["crop_bbox"]
        z0, z1 = bbox["z0"], bbox["z1"]
        y0, y1 = bbox["y0"], bbox["y1"]
        x0, x1 = bbox["x0"], bbox["x1"]
        bbox_shape = (z1 - z0, y1 - y0, x1 - x0)

        # Resize cam_crop to match exact bbox size if minor mismatch
        if cam_crop.shape != bbox_shape:
            factors = np.array(bbox_shape) / np.array(cam_crop.shape)
            cam_crop = zoom(cam_crop, factors, order=1).astype(np.float32)
            lo2, hi2 = cam_crop.min(), cam_crop.max()
            if hi2 > lo2:
                cam_crop = (cam_crop - lo2) / (hi2 - lo2)

        # Embed into full-volume-sized array
        cam_full = np.zeros(full_volume_shape, dtype=np.float32)
        # Clamp bbox to actual volume dims (safety)
        vz, vy, vx = full_volume_shape
        ez0, ez1 = min(z0, vz), min(z1, vz)
        ey0, ey1 = min(y0, vy), min(y1, vy)
        ex0, ex1 = min(x0, vx), min(x1, vx)
        cz = ez1 - ez0
        cy = ey1 - ey0
        cx = ex1 - ex0
        cam_full[ez0:ez1, ey0:ey1, ex0:ex1] = cam_crop[:cz, :cy, :cx]

        return cam_full, probs, pred, true_label

    except Exception as e:
        print(f"  GradCAM load failed for {pid}_{sid}: {e}")
        return None, None, None, None


def make_gradcam_overlay(ct_slice, cam_slice):
    """Blend CT slice with GradCAM heatmap (JET colourmap)."""
    u8 = (ct_slice * 255).astype(np.uint8)
    rgb = np.stack([u8] * 3, axis=-1)
    jet = cv2.applyColorMap((cam_slice * 255).astype(np.uint8), cv2.COLORMAP_JET)
    jet = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(rgb, 0.55, jet, 0.45, 0)

In [ ]:
# ================================================================
#  Metadata formatting
# ================================================================

def _fmt_dicom_time(raw):
    s = str(raw).strip()
    if len(s) >= 6:
        return f"{s[0:2]}:{s[2:4]}:{s[4:6]}"
    return s


def _fmt_dicom_date(raw):
    s = str(raw).strip()
    if len(s) >= 8:
        return f"{s[0:4]}-{s[4:6]}-{s[6:8]}"
    return s

In [ ]:
# ================================================================
#  MAIN VIEWER
# ================================================================

def rsna_full_viewer(root_path, masks_path, gradcam_dir, series_csv,
                     crop_meta_path, center=50, width=400):

    meta_df = pd.read_csv(series_csv)
    meta_df["patient_id"] = meta_df["patient_id"].astype(str)
    meta_df["series_id"] = meta_df["series_id"].astype(str)

    # Load crop metadata once for GradCAM bbox embedding
    crop_lookup = _load_crop_metadata(crop_meta_path)
    print(f"Crop metadata loaded: {len(crop_lookup)} entries")

    # -- Filter dropdowns
    spleen_dd = Dropdown(description="Spleen",
                         options=["Any", "Healthy", "Low", "High"],
                         value="Any", layout={"width": "180px"})
    liver_dd = Dropdown(description="Liver",
                        options=["Any", "Healthy", "Low", "High"],
                        value="Any", layout={"width": "180px"})
    kidney_dd = Dropdown(description="Kidney",
                         options=["Any", "Healthy", "Low", "High"],
                         value="Any", layout={"width": "190px"})
    bowel_dd = Dropdown(description="Bowel",
                        options=["Any", "Healthy", "Injury"],
                        value="Any", layout={"width": "180px"})
    extra_dd = Dropdown(description="Extravasation",
                        options=["Any", "Healthy", "Injury"],
                        value="Any", layout={"width": "220px"})

    patient_dd = Dropdown(description="Patient:")
    series_dd = Dropdown(description="Series:")
    slider = IntSlider(description="Slice")
    save_btn = Button(description="Save Slice", button_style="success")

    labels_out = Output()
    img_out = Output()
    save_out = Output()

    # -- Filtering logic
    def apply_filter():
        t = meta_df.copy()
        if spleen_dd.value == "Low":
            t = t[t.spleen_low == 1]
        elif spleen_dd.value == "High":
            t = t[t.spleen_high == 1]
        elif spleen_dd.value == "Healthy":
            t = t[(t.spleen_low == 0) & (t.spleen_high == 0)]

        for organ, dd in [("liver", liver_dd), ("kidney", kidney_dd)]:
            lo_col, hi_col = f"{organ}_low", f"{organ}_high"
            if lo_col in t.columns and hi_col in t.columns:
                if dd.value == "Low":
                    t = t[t[lo_col] == 1]
                elif dd.value == "High":
                    t = t[t[hi_col] == 1]
                elif dd.value == "Healthy":
                    t = t[(t[lo_col] == 0) & (t[hi_col] == 0)]

        if "extravasation_injury" in t.columns:
            if extra_dd.value == "Injury":
                t = t[t.extravasation_injury == 1]
            elif extra_dd.value == "Healthy":
                t = t[t.extravasation_injury == 0]

        if "bowel_injury" in t.columns:
            if bowel_dd.value == "Injury":
                t = t[t.bowel_injury == 1]
            elif bowel_dd.value == "Healthy":
                t = t[t.bowel_injury == 0]

        return t

    def update_patients(change):
        filtered = apply_filter()
        if filtered.empty:
            patient_dd.options = []
            with labels_out:
                labels_out.clear_output()
                print("No matching patients")
            return
        pts = sorted(filtered.patient_id.unique().tolist())
        patient_dd.options = pts
        patient_dd.value = pts[0]

    # -- Show metadata
    def show_metadata(pid, sid):
        with labels_out:
            labels_out.clear_output()
            row = meta_df[(meta_df.patient_id == str(pid)) &
                          (meta_df.series_id == str(sid))]
            if row.empty:
                row = meta_df[meta_df.patient_id == str(pid)]
            if row.empty:
                print(f"Patient {pid} -- no metadata found")
                return
            r = row.iloc[0]

            age = r.get("Age", "?")
            sex = r.get("Sex", "?")
            date = _fmt_dicom_date(r.get("Date", ""))
            t0 = _fmt_dicom_time(r.get("Start time", ""))
            t1 = _fmt_dicom_time(r.get("End time", ""))
            injury = r.get("Injury name", "")
            injury_str = injury if pd.notna(injury) and injury != "" else "None"
            aortic = r.get("aortic_hu", "?")

            print(f"===  Patient {pid}  |  Series {sid}  ===")
            print(f"  Age : {age}     Sex : {sex}")
            print(f"  Date: {date}    Time: {t0} -> {t1}")
            print(f"  Aortic HU: {aortic}")
            print(f"  Injury   : {injury_str}")

            sp_parts = []
            if r.get("spleen_healthy", 0) == 1:
                sp_parts.append("Healthy")
            if r.get("spleen_low", 0) == 1:
                sp_parts.append("Low")
            if r.get("spleen_high", 0) == 1:
                sp_parts.append("High")
            print(f"  Spleen   : {', '.join(sp_parts) if sp_parts else '--'}")

            if r.get("extravasation_injury", 0) == 1:
                print("  Extravasation: Injury")
            elif r.get("extravasation_healthy", 0) == 1:
                print("  Extravasation: Healthy")

    def update_series(change):
        pid = patient_dd.value
        if pid is None:
            return
        patient_dir = os.path.join(root_path, "train_images", pid)
        if not os.path.isdir(patient_dir):
            return
        series = sorted(os.listdir(patient_dir))
        series_dd.options = series
        if series:
            series_dd.value = series[0]
        load_series(None)

    # -- Load volume + mask + GradCAM
    def load_series(change):
        pid = patient_dd.value
        sid = series_dd.value
        if pid is None or sid is None:
            return

        show_metadata(pid, sid)

        series_dir = os.path.join(root_path, "train_images", pid, sid)
        volume = load_dicom_series_fast(series_dir, center, width)

        # Spleen mask
        mask = None
        mask_dir = os.path.join(masks_path, str(pid), str(sid))
        if os.path.isdir(mask_dir):
            mask_files = [f for f in os.listdir(mask_dir)
                          if f.endswith(".nii.gz") or f.endswith("nii_gz")]
            if mask_files:
                try:
                    mask = load_mask(os.path.join(mask_dir, mask_files[0]),
                                     volume.shape)
                except Exception as e:
                    print(f"  Mask load failed: {e}")

        # GradCAM -- embed into full volume via crop bbox
        cam, probs, pred, true_lbl = load_gradcam(
            gradcam_dir, crop_lookup, pid, sid, volume.shape)

        has_mask = mask is not None
        has_cam = cam is not None
        n_cols = 1 + int(has_mask) + int(has_cam) * 2

        slider.min = 0
        slider.max = volume.shape[0] - 1
        slider.value = volume.shape[0] // 2

        def _draw_mask_overlay(ax, ct_sl, mask_sl):
            ax.imshow(ct_sl, cmap="gray")
            if mask_sl is not None and mask_sl.any():
                rgba = np.zeros((*mask_sl.shape, 4), dtype=np.float32)
                rgba[mask_sl == 1] = [0.0, 1.0, 0.2, 0.35]
                ax.imshow(rgba)
                ax.contour(mask_sl, levels=[0.5], colors="lime", linewidths=1.5)

        def update_slice(change):
            z = slider.value
            with img_out:
                img_out.clear_output(wait=True)
                fig_w = 5 * n_cols
                fig, axes = plt.subplots(
                    1, n_cols, figsize=(fig_w, 5), facecolor="#111111")
                if n_cols == 1:
                    axes = [axes]

                col = 0
                axes[col].imshow(volume[z], cmap="gray")
                axes[col].set_title(
                    f"CT  (slice {z+1}/{volume.shape[0]})",
                    color="white", fontsize=9)
                axes[col].axis("off")
                col += 1

                if has_mask:
                    _draw_mask_overlay(axes[col], volume[z], mask[z])
                    axes[col].set_title("Spleen Mask",
                                        color="white", fontsize=9)
                    axes[col].axis("off")
                    col += 1

                if has_cam:
                    overlay = make_gradcam_overlay(volume[z], cam[z])
                    axes[col].imshow(overlay)
                    axes[col].set_title("GradCAM Overlay",
                                        color="white", fontsize=9)
                    axes[col].axis("off")
                    col += 1

                    axes[col].imshow(cam[z], cmap="jet", vmin=0, vmax=1)
                    axes[col].set_title("GradCAM",
                                        color="white", fontsize=9)
                    axes[col].axis("off")
                    col += 1

                title_parts = [f"Patient {pid}  |  Series {sid}"]
                if has_cam:
                    correct = pred == true_lbl
                    title_parts.append(
                        f"True: {CLASS_NAMES[true_lbl]}   "
                        f"Pred: {CLASS_NAMES[pred]}  "
                        f"{'CORRECT' if correct else 'WRONG'}   "
                        f"P(H)={probs[0]:.3f}  "
                        f"P(L)={probs[1]:.3f}  "
                        f"P(Hi)={probs[2]:.3f}")
                    title_color = "#2ecc71" if correct else "#e74c3c"
                else:
                    title_color = "white"

                fig.suptitle("\n".join(title_parts),
                             fontsize=10, fontweight="bold",
                             color=title_color)
                plt.tight_layout()
                plt.show()

        slider.observe(update_slice, names="value")
        update_slice(None)

        def save_slice(b):
            z = slider.value
            fname = f"{pid}_{sid}_slice{z+1}_{uuid.uuid4().hex[:6]}.png"
            n = n_cols
            fig, axes = plt.subplots(1, n, figsize=(5 * n, 5),
                                     facecolor="#111111")
            if n == 1:
                axes = [axes]
            c = 0
            axes[c].imshow(volume[z], cmap="gray")
            axes[c].axis("off")
            c += 1
            if has_mask:
                _draw_mask_overlay(axes[c], volume[z], mask[z])
                axes[c].axis("off")
                c += 1
            if has_cam:
                axes[c].imshow(make_gradcam_overlay(volume[z], cam[z]))
                axes[c].axis("off")
                c += 1
                axes[c].imshow(cam[z], cmap="jet", vmin=0, vmax=1)
                axes[c].axis("off")
                c += 1
            fig.savefig(fname, bbox_inches="tight",
                        pad_inches=0, facecolor="#111111")
            plt.close(fig)
            with save_out:
                save_out.clear_output()
                print("Saved:")
                display(FileLink(fname))

        save_btn._click_handlers.callbacks = []
        save_btn.on_click(save_slice)

    for w in [spleen_dd, liver_dd, kidney_dd, bowel_dd, extra_dd]:
        w.observe(update_patients, names="value")
    patient_dd.observe(update_series, names="value")
    series_dd.observe(load_series, names="value")
    update_patients(None)

    display(VBox([
        HBox([spleen_dd, liver_dd, kidney_dd, bowel_dd, extra_dd]),
        patient_dd, series_dd, labels_out,
        HBox([slider, save_btn]),
        img_out, save_out,
    ]))

In [ ]:
# Soft-tissue window (WC=50, WW=400)
rsna_full_viewer(ROOT_PATH, MASKS_PATH, GRADCAM_DIR, SERIES_CSV, CROP_META,
                 center=50, width=400)

Crop metadata loaded: 4684 entries


In [ ]:
# Wider window (WC=50, WW=800)
rsna_full_viewer(ROOT_PATH, MASKS_PATH, GRADCAM_DIR, SERIES_CSV, CROP_META,
                 center=50, width=800)

Crop metadata loaded: 4684 entries
